In [ ]:
"""
台股BIGRU預測系統 - 基準模型
用於與HFSLS-PSO-BIGRU進行消融實驗對比
"""

import math
import os
import pandas as pd
import numpy as np
from collections import deque
from keras.models import Sequential
from keras.layers import Dense, Dropout, GRU, Bidirectional
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, mean_absolute_percentage_error
import warnings
import time
from datetime import timedelta
warnings.filterwarnings('ignore')
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

# ============================================================
# 全局配置
# ============================================================
scaler = MinMaxScaler()
window = 30
loss = "mse"
measure = [[]]
true_pre = [[], []]
DATA_PATH = './data/' # 建議使用相對路徑以利 GitHub 使用

# ============================================================
# 核心函數
# ============================================================

def process_data(X):
    que = deque(maxlen=window)
    x = []
    for i in X:
        que.append(i)
        if len(que) == window:
            x.append(list(que))
    return np.array(x)

def build_baseline_bigru(input_shape, units=64, dropout=0.2):
    model = Sequential([
        Bidirectional(GRU(units, activation='relu', return_sequences=False), input_shape=input_shape),
        Dropout(dropout),
        Dense(1)
    ])
    model.compile(optimizer='adam', loss='mse', metrics=['mse'])
    return model

def train_baseline_model(x_train, y_train, x_test, y_test, test_indices, dates, stock_name):
    print(f"   構建基準BIGRU模型...")
    print(f"   參數：units=64, dropout=0.2, epochs=30")
    
    model = build_baseline_bigru(input_shape=x_train.shape[1:], units=64, dropout=0.2)
    model.fit(x_train, y_train, batch_size=32, epochs=30, validation_split=0.1, verbose=0, shuffle=False)
    
    y_pred = model.predict(x_test, verbose=0)
    mse = mean_squared_error(y_test, y_pred)
    measure[0] = [mse, math.sqrt(mse), mean_absolute_error(y_test, y_pred), mean_absolute_percentage_error(y_test, y_pred), r2_score(y_test, y_pred)]
    
    y_pred_inv = scaler.inverse_transform(y_pred)
    y_test_inv = scaler.inverse_transform(y_test)
    
    # 保存預測結果
    data = pd.DataFrame({
        'stock_name': stock_name, 'predict_date': [dates[idx] for idx in test_indices],
        'predicted_close': y_pred_inv[:, 0], 'actual_close': y_test_inv[:, 0]
    })
    
    output_path = os.path.join(DATA_PATH, 'predicted_baseline_bigru.csv')
    os.makedirs(os.path.dirname(output_path), exist_ok=True)
    
    if os.path.exists(output_path):
        existing = pd.read_csv(output_path)
        existing = existing[existing['stock_name'] != stock_name]
        existing = pd.concat([existing, data], ignore_index=True)
    else: existing = data
    existing.to_csv(output_path, index=False, encoding='utf-8-sig')
    return model

def main(data_file, stock_name):
    global measure
    df = pd.read_csv(os.path.join(DATA_PATH, data_file)).fillna(0).sort_index()
    dates = pd.to_datetime(df['Date']) if 'Date' in df.columns else pd.date_range('2021-01-01', periods=len(df), freq='D')
    
    X = df.drop(columns=['Date'], errors='ignore').iloc[:, :-1].apply(lambda x: (x - x.min()) / (x.max() - x.min() + 1e-7))
    y = scaler.fit_transform(df['Close'].values[:-window+1].reshape(-1, 1))
    x = process_data(X.values)
    
    split_point = int(len(y) * 0.7)
    train_baseline_model(x[:split_point], y[:split_point], x[split_point:], y[split_point:], 
                         list(range(split_point, len(y))), dates[:-window+1].reset_index(drop=True), stock_name)

if __name__ == '__main__':
    print("======================================================================")
    print("台股BIGRU預測系統 - 基準模型")
    print("======================================================================")
    
    stock_list = ['台積電', '長榮', '聯發科', '統一超']
    all_results = []
    for stock in stock_list:
        main(f"{stock}_date.csv", stock)
        all_results.append({'stock': stock, 'MSE': measure[0][0], 'RMSE': measure[0][1], 'R2': measure[0][4]})
    
    # 總結表格
    print(f"\n性能總結")
    print(f"{'股票':<10} {'MSE':<12} {'RMSE':<12} {'R2':<10}")
    for r in all_results:
        print(f"{r['stock']:<10} {r['MSE']:<12.6f} {r['RMSE']:<12.6f} {r['R2']:<10.4f}")